# ASC Phase 2

- Gold nằm trực tiếp trong `/content/drive/MyDrive/asc_phase2/gold_train.csv` và `gold_test.csv`
- High confidence nằm trong `/content/drive/MyDrive/asc_phase2/raw_high_conf`
- High confidence là parquet có `aspects: array<string>`, `sentiments: array<array<double>>`
- Gold là CSV nên `aspects`, `sentiments` bị đọc thành string. Code sẽ parse lại.
- Label lấy từ `argmax(sentiments)` với `0=negative, 1=neutral, 2=positive`
- Sample 400k pair/category theo rule: lấy toàn bộ neutral, còn lại pos/neg 1:1
- Merge gold train, remove leakage với gold test


In [ ]:
!pip -q install pyspark==3.5.1 gdown pyarrow pandas datasets transformers accelerate evaluate scikit-learn sentencepiece

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.0/317.0 MB 4.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.1.0 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.1 which is incompatible.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, ast
from pathlib import Path

RANDOM_SEED = 2026
N_PER_CATEGORY = 400_000

PROJECT_DIR = "/content/drive/MyDrive/asc_phase2"
RAW_DIR = f"{PROJECT_DIR}/raw_high_conf"
OUTPUT_DIR = f"{PROJECT_DIR}/outputs"
MODEL_DIR = f"{PROJECT_DIR}/asc_teacher_phase1/asc_teacher_phase1"

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

PSEUDO_SAMPLED_BY_CAT_PATH = f"{OUTPUT_DIR}/pseudo_sampled_by_category"
PSEUDO_SAMPLED_PATH = f"{OUTPUT_DIR}/pseudo_sampled_2m"
FINAL_TRAIN_PATH = f"{OUTPUT_DIR}/final_train_no_leakage"
REPORT_PATH = f"{OUTPUT_DIR}/sampling_report.json"
TEST_METRICS_PATH = f"{OUTPUT_DIR}/gold_test_metrics.json"

# Gold file thật đang nằm ngay trong asc_phase2
GOLD_TRAIN_PATH = f"{PROJECT_DIR}/gold_train.csv"
GOLD_TEST_PATH = f"{PROJECT_DIR}/gold_test.csv"

HIGH_CONF_GDRIVE_IDS = {
    "electronics_p1" : "1nDYriP7FgAS8gCxQYGMPwGkwOuWM6mRR",
    "electronics_p2" : "1I1Vn29wx5RQ28BqXudbQphikNEbMThgE",
    "software"       : "1L1Q5zYY_XGUiPclTrap4ssnH8nhKUY8R",
    "kindle_store"   : "1H4QWm_eR37k3h2mivEjWa1j3HP_zBlkr",
    "office_products": "1lvpFGj-zYomh7y2rMNoboqCHNYloAu0s",
}

ASC_HIGH_PATHS = {name: f"{RAW_DIR}/{name}.parquet" for name in HIGH_CONF_GDRIVE_IDS}

print("PROJECT_DIR:", PROJECT_DIR)
print("RAW_DIR:", RAW_DIR)
print("GOLD_TRAIN_PATH:", GOLD_TRAIN_PATH)
print("GOLD_TEST_PATH:", GOLD_TEST_PATH)
ASC_HIGH_PATHS


Mounted at /content/drive
PROJECT_DIR: /content/drive/MyDrive/asc_phase2
RAW_DIR: /content/drive/MyDrive/asc_phase2/raw_high_conf
GOLD_TRAIN_PATH: /content/drive/MyDrive/asc_phase2/gold_train.csv
GOLD_TEST_PATH: /content/drive/MyDrive/asc_phase2/gold_test.csv


{'electronics_p1': '/content/drive/MyDrive/asc_phase2/raw_high_conf/electronics_p1.parquet',
 'electronics_p2': '/content/drive/MyDrive/asc_phase2/raw_high_conf/electronics_p2.parquet',
 'software': '/content/drive/MyDrive/asc_phase2/raw_high_conf/software.parquet',
 'kindle_store': '/content/drive/MyDrive/asc_phase2/raw_high_conf/kindle_store.parquet',
 'office_products': '/content/drive/MyDrive/asc_phase2/raw_high_conf/office_products.parquet'}

In [ ]:
import gdown, os

def download_gdrive_file(file_id: str, output_path: str, overwrite: bool = False):
    if os.path.exists(output_path) and not overwrite:
        print(f"[EXISTS] {output_path}")
        return output_path
    url = f"https://drive.google.com/uc?id={file_id}"
    print(f"[DOWNLOAD] {file_id} -> {output_path}")
    gdown.download(url, output_path, quiet=False, fuzzy=True)
    return output_path

for name, file_id in HIGH_CONF_GDRIVE_IDS.items():
    download_gdrive_file(file_id, ASC_HIGH_PATHS[name], overwrite=False)


[EXISTS] /content/drive/MyDrive/asc_phase2/raw_high_conf/electronics_p1.parquet
[EXISTS] /content/drive/MyDrive/asc_phase2/raw_high_conf/electronics_p2.parquet
[EXISTS] /content/drive/MyDrive/asc_phase2/raw_high_conf/software.parquet
[EXISTS] /content/drive/MyDrive/asc_phase2/raw_high_conf/kindle_store.parquet
[EXISTS] /content/drive/MyDrive/asc_phase2/raw_high_conf/office_products.parquet


In [ ]:
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, StringType, DoubleType

spark = (
    SparkSession.builder
    .appName("ASC Phase 2 Final Fixed")
    .config("spark.driver.memory", "10g")
    .config("spark.sql.shuffle.partitions", "64")
    .config("spark.sql.files.maxPartitionBytes", "64m")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark:", spark.version)


Spark: 3.5.1


In [ ]:
LABEL2ID = {"negative": 0, "neutral": 1, "positive": 2}
ID2LABEL = {0: "negative", 1: "neutral", 2: "positive"}

def read_table_spark(path: str) -> DataFrame:
    if path.endswith(".parquet"):
        return spark.read.parquet(path)
    if path.endswith(".csv"):
        return (
            spark.read
            .option("header", True)
            .option("multiLine", True)
            .option("quote", '"')
            .option("escape", '"')
            .option("mode", "PERMISSIVE")
            .csv(path)
        )
    raise ValueError(f"Only parquet/csv supported: {path}")

def first_existing_col(df: DataFrame, candidates: list[str]) -> str | None:
    cols = set(df.columns)
    for c in candidates:
        if c in cols:
            return c
    return None

def make_sample_key(sentence_col="sentence", aspect_col="aspect"):
    return F.sha2(
        F.concat_ws(
            "||",
            F.lower(F.trim(F.col(sentence_col))),
            F.lower(F.trim(F.col(aspect_col))),
        ),
        256,
    )

def _parse_aspects_py(x):
    if x is None:
        return []
    if isinstance(x, list):
        return [str(v) for v in x if v is not None]
    s = str(x).strip()
    if s == "" or s == "[]":
        return []
    try:
        v = ast.literal_eval(s)
        if isinstance(v, list):
            return [str(i) for i in v if i is not None]
    except Exception:
        pass
    try:
        v = json.loads(s.replace("'", '"'))
        if isinstance(v, list):
            return [str(i) for i in v if i is not None]
    except Exception:
        pass
    return []

def _parse_sentiments_py(x):
    if x is None:
        return []
    if isinstance(x, list):
        return x
    s = str(x).strip()
    if s == "" or s == "[]":
        return []
    try:
        v = ast.literal_eval(s)
        if isinstance(v, list):
            out = []
            for row in v:
                if isinstance(row, list) and len(row) >= 3:
                    out.append([float(row[0]), float(row[1]), float(row[2])])
            return out
    except Exception:
        pass
    try:
        v = json.loads(s.replace("'", '"'))
        if isinstance(v, list):
            out = []
            for row in v:
                if isinstance(row, list) and len(row) >= 3:
                    out.append([float(row[0]), float(row[1]), float(row[2])])
            return out
    except Exception:
        pass
    return []

parse_aspects_udf = F.udf(_parse_aspects_py, ArrayType(StringType()))
parse_sentiments_udf = F.udf(_parse_sentiments_py, ArrayType(ArrayType(DoubleType())))

def fix_array_string_columns(df: DataFrame, aspect_col: str, sentiment_col: str) -> DataFrame:
    dtypes = dict(df.dtypes)
    if dtypes.get(aspect_col) == "string":
        df = df.withColumn(aspect_col, parse_aspects_udf(F.col(aspect_col)))
    if dtypes.get(sentiment_col) == "string":
        df = df.withColumn(sentiment_col, parse_sentiments_udf(F.col(sentiment_col)))
    return df

def flatten_arrays_to_pairs(df: DataFrame, source_name: str) -> DataFrame:
    sentence_col = first_existing_col(df, ["sentence_text", "sentence", "text", "review_sentence", "review_text", "content"])
    aspect_col = first_existing_col(df, ["aspects", "aspect_terms", "aspect_list", "targets", "terms"])
    sentiment_col = first_existing_col(df, ["sentiments", "sentiment_probs", "probabilities", "probs", "scores"])

    if sentence_col is None:
        raise ValueError(f"No sentence column. Columns: {df.columns}")
    if aspect_col is None:
        raise ValueError(f"No aspects column. Columns: {df.columns}")
    if sentiment_col is None:
        raise ValueError(f"No sentiments column. Columns: {df.columns}")

    df = fix_array_string_columns(df, aspect_col, sentiment_col)

    parent_expr = F.col("parent_asin").cast("string") if "parent_asin" in df.columns else F.lit(None).cast("string")
    sid_expr = F.col("sentence_id").cast("string") if "sentence_id" in df.columns else F.lit(None).cast("string")
    rating_expr = F.col("rating").cast("double") if "rating" in df.columns else F.lit(None).cast("double")
    gate_expr = F.col("gate_confidence").cast("double") if "gate_confidence" in df.columns else F.lit(None).cast("double")

    pair_df = (
        df
        .withColumn("source_category", F.lit(source_name))
        .withColumn("_zipped", F.arrays_zip(F.col(aspect_col), F.col(sentiment_col)))
        .withColumn("_pair", F.explode_outer(F.col("_zipped")))
        .select(
            F.col("source_category").alias("category_name"),
            parent_expr.alias("parent_asin"),
            sid_expr.alias("sentence_id"),
            F.col(sentence_col).cast("string").alias("sentence"),
            F.col(f"_pair.{aspect_col}").cast("string").alias("aspect"),
            F.col(f"_pair.{sentiment_col}").alias("sentiment_probs"),
            rating_expr.alias("rating"),
            gate_expr.alias("gate_confidence"),
        )
        .filter(F.col("sentence").isNotNull())
        .filter(F.col("aspect").isNotNull())
        .filter(F.col("sentiment_probs").isNotNull())
        .filter(F.length(F.trim(F.col("sentence"))) > 0)
        .filter(F.length(F.trim(F.col("aspect"))) > 0)
        .withColumn("sentence", F.trim(F.col("sentence")))
        .withColumn("aspect", F.trim(F.col("aspect")))
        .withColumn("prob_neg", F.col("sentiment_probs")[0].cast("double"))
        .withColumn("prob_neu", F.col("sentiment_probs")[1].cast("double"))
        .withColumn("prob_pos", F.col("sentiment_probs")[2].cast("double"))
        .filter(F.col("prob_neg").isNotNull())
        .filter(F.col("prob_neu").isNotNull())
        .filter(F.col("prob_pos").isNotNull())
        .withColumn(
            "label_id",
            F.when((F.col("prob_neg") >= F.col("prob_neu")) & (F.col("prob_neg") >= F.col("prob_pos")), F.lit(0))
             .when((F.col("prob_neu") >= F.col("prob_neg")) & (F.col("prob_neu") >= F.col("prob_pos")), F.lit(1))
             .otherwise(F.lit(2))
        )
        .withColumn(
            "label",
            F.when(F.col("label_id") == 0, F.lit("negative"))
             .when(F.col("label_id") == 1, F.lit("neutral"))
             .when(F.col("label_id") == 2, F.lit("positive"))
        )
        .withColumn("confidence", F.greatest(F.col("prob_neg"), F.col("prob_neu"), F.col("prob_pos")))
        .withColumn("sample_key", make_sample_key("sentence", "aspect"))
        .dropDuplicates(["sample_key", "label_id"])
    )

    return pair_df.select(
        "sample_key", "category_name", "parent_asin", "sentence_id",
        "sentence", "aspect", "label", "label_id",
        "confidence", "prob_neg", "prob_neu", "prob_pos",
        "rating", "gate_confidence",
    )


In [ ]:
# Test high-confidence + gold before running all

cat = "software"
df_high = spark.read.parquet(ASC_HIGH_PATHS[cat])
df_high_pair = flatten_arrays_to_pairs(df_high, cat)

print("High-conf pair count:", df_high_pair.count())
df_high_pair.show(10, truncate=False)
df_high_pair.groupBy("label_id", "label").count().orderBy("label_id").show()

gold_train_raw = read_table_spark(GOLD_TRAIN_PATH)
gold_test_raw = read_table_spark(GOLD_TEST_PATH)

print("GOLD TRAIN RAW SCHEMA")
gold_train_raw.printSchema()
gold_train_raw.select("aspects", "sentiments").show(20, truncate=False)

gold_train_pair_test = flatten_arrays_to_pairs(gold_train_raw, "gold_train")
gold_test_pair_test = flatten_arrays_to_pairs(gold_test_raw, "gold_test")

print("Gold train pair count:", gold_train_pair_test.count())
gold_train_pair_test.show(10, truncate=False)
gold_train_pair_test.groupBy("label_id", "label").count().orderBy("label_id").show()

print("Gold test pair count:", gold_test_pair_test.count())
gold_test_pair_test.show(10, truncate=False)
gold_test_pair_test.groupBy("label_id", "label").count().orderBy("label_id").show()


High-conf pair count: 2300103
+----------------------------------------------------------------+-------------+-----------+-----------+----------------------------------------------------------------------------------------+-----------------------+--------+--------+------------------+---------------------+---------------------+--------------------+------+------------------+
|sample_key                                                      |category_name|parent_asin|sentence_id|sentence                                                                                |aspect                 |label   |label_id|confidence        |prob_neg             |prob_neu             |prob_pos            |rating|gate_confidence   |
+----------------------------------------------------------------+-------------+-----------+-----------+----------------------------------------------------------------------------------------+-----------------------+--------+--------+------------------+---------------------+--

In [ ]:
def sample_exact_n_by_hash(df: DataFrame, n: int, seed: int, key_col: str = "sample_key") -> DataFrame:
    if n <= 0:
        return df.limit(0)

    total = df.count()
    if total <= n:
        return df

    fraction = min(1.0, (n * 1.5) / total)
    candidate = df.sample(False, fraction, seed=seed)

    return (
        candidate
        .withColumn("_rand_key", F.xxhash64(F.col(key_col), F.lit(seed)))
        .orderBy("_rand_key")
        .drop("_rand_key")
        .limit(n)
    )

def sample_phase2_category(df_pair: DataFrame, n_total: int = 400_000, seed: int = 42):
    neutral_df = df_pair.filter(F.col("label_id") == 1)
    pos_df = df_pair.filter(F.col("label_id") == 2)
    neg_df = df_pair.filter(F.col("label_id") == 0)

    n_neu_total = neutral_df.count()
    n_pos_total = pos_df.count()
    n_neg_total = neg_df.count()

    if n_neu_total >= n_total:
        sampled_neu = sample_exact_n_by_hash(neutral_df, n_total, seed)
        sampled_pos = pos_df.limit(0)
        sampled_neg = neg_df.limit(0)
        target_neu, target_pos, target_neg = n_total, 0, 0
    else:
        sampled_neu = neutral_df
        remaining = n_total - n_neu_total

        target_pos = remaining // 2
        target_neg = remaining - target_pos

        actual_target_pos = min(target_pos, n_pos_total)
        actual_target_neg = min(target_neg, n_neg_total)

        leftover = remaining - actual_target_pos - actual_target_neg

        if leftover > 0:
            add_pos = min(leftover, max(0, n_pos_total - actual_target_pos))
            actual_target_pos += add_pos
            leftover -= add_pos

        if leftover > 0:
            add_neg = min(leftover, max(0, n_neg_total - actual_target_neg))
            actual_target_neg += add_neg
            leftover -= add_neg

        target_neu = n_neu_total
        target_pos = actual_target_pos
        target_neg = actual_target_neg

        sampled_pos = sample_exact_n_by_hash(pos_df, int(target_pos), seed + 11)
        sampled_neg = sample_exact_n_by_hash(neg_df, int(target_neg), seed + 29)

    sampled = (
        sampled_neu
        .unionByName(sampled_pos, allowMissingColumns=True)
        .unionByName(sampled_neg, allowMissingColumns=True)
    )
    sampled = sample_exact_n_by_hash(sampled, n_total, seed + 101)

    final_counts = {
        ID2LABEL[int(r["label_id"])]: int(r["count"])
        for r in sampled.groupBy("label_id").count().collect()
    }

    stats = {
        "total_pairs_after_flatten": int(df_pair.count()),
        "available": {"negative": int(n_neg_total), "neutral": int(n_neu_total), "positive": int(n_pos_total)},
        "target": {"negative": int(target_neg), "neutral": int(target_neu), "positive": int(target_pos)},
        "sampled_pairs": int(sampled.count()),
        "final_label_counts": final_counts,
    }
    return sampled, stats


In [ ]:
# Run flatten + sample 400k/category

# Nếu chạy lại từ đầu, mở comment:
# !rm -rf "{PSEUDO_SAMPLED_BY_CAT_PATH}" "{PSEUDO_SAMPLED_PATH}" "{FINAL_TRAIN_PATH}"

sampled_paths = []
sampling_report = []

for category, path in ASC_HIGH_PATHS.items():
    print("\\n" + "=" * 100)
    print(f"[PROCESS] {category}")
    print(f"[PATH] {path}")

    raw_df = spark.read.parquet(path)
    df_pair = flatten_arrays_to_pairs(raw_df, category)

    print(f"[PAIR LEVEL] {category}: {df_pair.count():,}")
    df_pair.groupBy("label_id", "label").count().orderBy("label_id").show()

    sampled_cat, stats = sample_phase2_category(df_pair, n_total=N_PER_CATEGORY, seed=RANDOM_SEED)

    out_cat_path = f"{PSEUDO_SAMPLED_BY_CAT_PATH}/{category}"
    sampled_cat.repartition(4).write.mode("overwrite").parquet(out_cat_path)

    sampled_paths.append(out_cat_path)

    stats.update({"category_name": category, "input_path": path, "seed": RANDOM_SEED})
    sampling_report.append(stats)

    print(json.dumps(stats, ensure_ascii=False, indent=2))

with open(REPORT_PATH, "w", encoding="utf-8") as f:
    json.dump(sampling_report, f, ensure_ascii=False, indent=2)

print("Report:", REPORT_PATH)


\n====================================================================================================
[PROCESS] electronics_p1
[PATH] /content/drive/MyDrive/asc_phase2/raw_high_conf/electronics_p1.parquet
[PAIR LEVEL] electronics_p1: 15,416,279
+--------+--------+--------+
|label_id|   label|   count|
+--------+--------+--------+
|       0|negative| 4529801|
|       1| neutral|   13672|
|       2|positive|10872806|
+--------+--------+--------+

{
  "total_pairs_after_flatten": 15416279,
  "available": {
    "negative": 4529801,
    "neutral": 13672,
    "positive": 10872806
  },
  "target": {
    "negative": 193164,
    "neutral": 13672,
    "positive": 193164
  },
  "sampled_pairs": 400000,
  "final_label_counts": {
    "neutral": 13672,
    "positive": 193164,
    "negative": 193164
  },
  "category_name": "electronics_p1",
  "input_path": "/content/drive/MyDrive/asc_phase2/raw_high_conf/electronics_p1.parquet",
  "seed": 2026
}
\n====================================================

In [ ]:
# Merge pseudo sampled

pseudo_df = spark.read.parquet(*sampled_paths)
pseudo_df = pseudo_df.dropDuplicates(["sample_key", "label_id"])

pseudo_df.repartition(8).write.mode("overwrite").parquet(PSEUDO_SAMPLED_PATH)

print("Saved pseudo:", PSEUDO_SAMPLED_PATH)
print("Pseudo count:", pseudo_df.count())
pseudo_df.groupBy("category_name").count().orderBy("category_name").show(100, truncate=False)
pseudo_df.groupBy("label_id", "label").count().orderBy("label_id").show(100, truncate=False)


Saved pseudo: /content/drive/MyDrive/asc_phase2/outputs/pseudo_sampled_2m
Pseudo count: 1997451
+---------------+------+
|category_name  |count |
+---------------+------+
|electronics_p1 |399917|
|electronics_p2 |398296|
|kindle_store   |400000|
|office_products|399273|
|software       |399965|
+---------------+------+

+--------+--------+------+
|label_id|label   |count |
+--------+--------+------+
|0       |negative|981265|
|1       |neutral |34817 |
|2       |positive|981369|
+--------+--------+------+



In [ ]:
# Gold train/test flatten + leakage removal + final train

gold_train_raw = read_table_spark(GOLD_TRAIN_PATH)
gold_test_raw = read_table_spark(GOLD_TEST_PATH)

gold_train_df = (
    flatten_arrays_to_pairs(gold_train_raw, "gold_train")
    .select("sample_key", "category_name", "sentence", "aspect", "label", "label_id")
)

gold_test_df = (
    flatten_arrays_to_pairs(gold_test_raw, "gold_test")
    .select("sample_key", "category_name", "sentence", "aspect", "label", "label_id")
)

print("Gold train pair-level:", gold_train_df.count())
print("Gold test pair-level:", gold_test_df.count())
gold_train_df.groupBy("label_id", "label").count().orderBy("label_id").show()
gold_test_df.groupBy("label_id", "label").count().orderBy("label_id").show()

gold_test_keys = gold_test_df.select("sample_key").dropDuplicates()

pseudo_no_leak = pseudo_df.join(gold_test_keys, on="sample_key", how="left_anti")
gold_train_no_leak = gold_train_df.join(gold_test_keys, on="sample_key", how="left_anti")

final_train_df = (
    pseudo_no_leak
    .select("sample_key", "category_name", "sentence", "aspect", "label", "label_id")
    .unionByName(
        gold_train_no_leak.select("sample_key", "category_name", "sentence", "aspect", "label", "label_id"),
        allowMissingColumns=True,
    )
    .dropDuplicates(["sample_key", "label_id"])
)

final_train_df.repartition(8).write.mode("overwrite").parquet(FINAL_TRAIN_PATH)

print("Pseudo before leakage removal:", pseudo_df.count())
print("Pseudo no leak:", pseudo_no_leak.count())
print("Gold train before leakage removal:", gold_train_df.count())
print("Gold train no leak:", gold_train_no_leak.count())
print("Final train:", final_train_df.count())
final_train_df.groupBy("label_id", "label").count().orderBy("label_id").show()


Gold train pair-level: 2660
Gold test pair-level: 681
+--------+--------+-----+
|label_id|   label|count|
+--------+--------+-----+
|       0|negative|  831|
|       1| neutral|  135|
|       2|positive| 1694|
+--------+--------+-----+

+--------+--------+-----+
|label_id|   label|count|
+--------+--------+-----+
|       0|negative|  217|
|       1| neutral|   29|
|       2|positive|  435|
+--------+--------+-----+

Pseudo before leakage removal: 1997451
Pseudo no leak: 1997430
Gold train before leakage removal: 2660
Gold train no leak: 2655
Final train: 2000001
+--------+--------+------+
|label_id|   label| count|
+--------+--------+------+
|       0|negative|982038|
|       1| neutral| 34948|
|       2|positive|983015|
+--------+--------+------+



In [ ]:
from datasets import load_dataset, DatasetDict, ClassLabel, Features, Value

GOLD_TEST_STD_PATH = f"{OUTPUT_DIR}/gold_test_standardized"
gold_test_df.repartition(1).write.mode("overwrite").parquet(GOLD_TEST_STD_PATH)

hf_data = load_dataset(
    "parquet",
    data_files={
        "train": f"{FINAL_TRAIN_PATH}/*.parquet",
        "test": f"{GOLD_TEST_STD_PATH}/*.parquet",
    },
)

# Define features for the output of to_model_example
# LABEL2ID and ID2LABEL are already defined in a previous cell
num_classes = len(LABEL2ID)
output_features = Features({
    "sentence": Value("string"),
    "aspect": Value("string"),
    "labels": ClassLabel(num_classes=num_classes, names=list(LABEL2ID.keys())),
})

def to_model_example(ex):
    return {
        "sentence": str(ex["sentence"]),
        "aspect": str(ex["aspect"]),
        "labels": int(ex["label_id"]),
    }

train_hf = hf_data["train"].map(
    to_model_example,
    remove_columns=hf_data["train"].column_names,
    features=output_features
)
test_hf = hf_data["test"].map(
    to_model_example,
    remove_columns=hf_data["test"].column_names,
    features=output_features
)

dataset = DatasetDict({"train": train_hf, "test": test_hf})

split = dataset["train"].train_test_split(
    test_size=0.02,
    seed=RANDOM_SEED,
    stratify_by_column="labels",
)

dataset = DatasetDict({
    "train": split["train"],
    "validation": split["test"],
    "test": dataset["test"],
})

dataset

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/2000001 [00:00<?, ? examples/s]

Map:   0%|          | 0/681 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence', 'aspect', 'labels'],
        num_rows: 1960000
    })
    validation: Dataset({
        features: ['sentence', 'aspect', 'labels'],
        num_rows: 40001
    })
    test: Dataset({
        features: ['sentence', 'aspect', 'labels'],
        num_rows: 681
    })
})